# Pareto Frontier (Figure 3 c, d, e)

Three-panel Pareto figure comparing CLP-SNN on Intel Loihi 2 against four
baselines (CLP, NCM, Replay, Streaming-LDA periodic with k=1) on a Jetson
Orin Nano CPU and GPU.

**Panel layout:**
- **(c)** Accuracy vs. Learning Energy Efficiency (Hz/J), log-y
- **(d)** Accuracy vs. Max Learning Frequency (Hz), log-y
- **(e)** Frequency vs. Energy Efficiency (log-log), with regression
  line through non-Loihi baselines

## Data sources

**Latency and energy** values are from the revised post-rebuttal benchmark
(`table_1_revision.xlsx`, sheet "final table"). The spreadsheet itself lives
outside the repository (Proton Drive); the relevant 9 rows are inlined below.

**Accuracy** values are unchanged from the original submission and were
measured on **Intel Loihi 2** using the proprietary Lava-INL toolchain
(Intel INRC, NDA-restricted). They cannot be regenerated from this repository
alone — see the manuscript for the experimental protocol.

The full reference implementation is in `analysis/pareto_plots.py`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns

REPO = Path.cwd().parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

IMAGES_DIR = REPO / 'images'
IMAGES_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'font.size': 7,
    'axes.labelsize': 7,
    'axes.titlesize': 7,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'lines.linewidth': 1,
    'axes.linewidth': 0.75,
    'xtick.major.size': 3,
    'ytick.major.size': 3,
    'xtick.major.width': 0.75,
    'ytick.major.width': 0.75,
    'savefig.dpi': 600,
    'svg.fonttype': 'none',
})

## Benchmark data

| # | Method | Device | Quant | Latency (ms) | Energy (mJ) | Acc 25-shot (%) |
|---|---|---|---|---|---|---|
| 1 | CLP-SNN | Loihi 2 | INT8 | 0.330 | 0.05 | 90.0 |
| 2 | CLP | CPU | fp32 | 1.076 | 8.66 | 93.0 |
| 3 | CLP | GPU | fp32 | 2.584 | 14.74 | 93.0 |
| 4 | NCM | CPU | fp32 | 0.338 | 1.85 | 84.5 |
| 5 | NCM | GPU | fp32 | 0.689 | 3.89 | 84.5 |
| 6 | Replay | CPU | fp32 | 6.584 | 50.92 | 91.6 |
| 7 | Replay | GPU | fp32 | 2.471 | 14.39 | 91.6 |
| 8 | SLDA-periodic(k=1) | CPU | fp32 | 73.196 | 677.84 | 95.7 |
| 9 | SLDA-periodic(k=1) | GPU | fp32 | 37.289 | 333.37 | 95.7 |

Latency and energy are taken from `table_1_revision.xlsx` (sheet "final table").
Accuracy values are the 25-shot results from Table 1 of the manuscript: the
Loihi 2 row was measured with the proprietary Lava-INL toolchain (Intel INRC
NDA), while baseline rows are PyTorch reference evaluations and are identical
across CPU/GPU rows of the same algorithm — the algorithm produces the same
predictions regardless of which device runs it; only latency and energy change.

In [ ]:
METHODS = ['CLP-SNN', 'CLP', 'CLP', 'ncm', 'ncm',
           'replay', 'replay', 'SLDA', 'SLDA']
DEVICES = ['Loihi 2', 'CPU', 'GPU', 'CPU', 'GPU',
           'CPU', 'GPU', 'CPU', 'GPU']

# Latency (ms) and energy (mJ) from table_1_revision.xlsx, sheet 'final table'
LATENCY_MS = np.array([0.33, 1.07555, 2.58396, 0.33803, 0.68864,
                       6.58365, 2.47102, 73.19625, 37.28893])
ENERGY_MJ  = np.array([0.05, 8.66, 14.74, 1.85, 3.89,
                       50.92, 14.39, 677.84, 333.37])

# 25-shot accuracy from Table 1 of the manuscript (CLP-SNN/Loihi: proprietary Lava-INL)
ACCURACY = np.array([90.0, 93.0, 93.0, 84.5, 84.5,
                     91.6, 91.6, 95.7, 95.7])

# Derived metrics
LATENCY_S         = LATENCY_MS / 1e3   # ms -> s
ENERGY_J          = ENERGY_MJ  / 1e3   # mJ -> J
THROUGHPUT        = 1.0 / LATENCY_S    # Hz
ENERGY_EFFICIENCY = 1.0 / ENERGY_J     # Hz/J

print(f'{len(METHODS)} data points loaded.')
print(f'  Throughput range:        {THROUGHPUT.min():.1f} - {THROUGHPUT.max():.1f} Hz')
print(f'  Energy efficiency range: {ENERGY_EFFICIENCY.min():.1f} - {ENERGY_EFFICIENCY.max():.1f} Hz/J')

## Three-panel Pareto figure

**Color** encodes algorithm: CLP / CLP-SNN (cyan), NCM (orange), Replay (green),
SLDA (red) — using `seaborn.color_palette('tab10')`.

**Marker** encodes architecture: `*` Loihi 2 (large), `D` GPU, `o` CPU.

**Regression lines** (dashed, blue) show the Pareto trend across baselines:
- Panels (c) and (d): fit through the 6 non-CLP baseline points
- Panel (e): log-log fit through the 8 non-Loihi baseline points

CLP-SNN on Loihi 2 sits clearly above/right of the regression line in all three
panels — that is the visual statement of Fig. 3 c/d/e.

In [ ]:
p = sns.color_palette('tab10')
# Color scheme matches experiments/clp_vs_baselines_1shot.py
# (CLP=p[9] cyan, NCM=p[2] green, Replay=p[3] red, SLDA=p[1] orange)
colors  = [p[9], p[9], p[9],  p[2], p[2],  p[3], p[3],  p[1], p[1]]
markers = ['*', 'o', 'D',  'o', 'D',  'o', 'D',  'o', 'D']

fig, axes = plt.subplots(figsize=(7.087, 3), ncols=3, nrows=1)
fig.patch.set_facecolor('white')

def scatter_all(ax, x, y):
    for i in range(len(x)):
        s = 50 if i == 0 else 9
        ax.scatter(x[i], y[i], color=colors[i], marker=markers[i], s=s)

# Panel C: Accuracy vs Energy Efficiency
ax = axes[0]
ax.set_yscale('log')
ax.set_xlim([80, 97])
ax.set_xlabel('Accuracy (%)')
ax.set_ylabel('Learning Energy Efficiency (Hz/J)')
ax.set_title('Energy Efficiency vs Accuracy')
scatter_all(ax, ACCURACY, ENERGY_EFFICIENCY)
log_y = np.log10(ENERGY_EFFICIENCY[3:])
coefs = np.polyfit(ACCURACY[3:], log_y, 1)
xs = np.linspace(ACCURACY[3:].min(), ACCURACY[3:].max(), 100)
ax.plot(xs, 10**(coefs[0]*xs + coefs[1]), 'b--', linewidth=0.4, alpha=0.8)
ax.grid(True, which='both', linestyle='--', linewidth=0.2)

# Panel D: Accuracy vs Throughput
ax = axes[1]
ax.set_yscale('log')
ax.set_xlim([80, 97])
ax.set_xlabel('Accuracy (%)')
ax.set_ylabel('Max Learning Frequency (Hz)')
ax.set_title('Throughput vs Accuracy')
scatter_all(ax, ACCURACY, THROUGHPUT)
log_y = np.log10(THROUGHPUT[3:])
coefs = np.polyfit(ACCURACY[3:], log_y, 1)
xs = np.linspace(ACCURACY[3:].min(), ACCURACY[3:].max(), 100)
ax.plot(xs, 10**(coefs[0]*xs + coefs[1]), 'b--', linewidth=0.4, alpha=0.8)
ax.grid(True, which='both', linestyle='--', linewidth=0.2)

# Panel E: Throughput vs Energy Efficiency (log-log)
ax = axes[2]
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim([10, 4000])
ax.set_ylim([1, 30000])
ax.set_xlabel('Max Learning Frequency (Hz)')
ax.set_ylabel('Learning Energy Efficiency (Hz/J)')
ax.set_title('Energy Efficiency & Throughput')
scatter_all(ax, THROUGHPUT, ENERGY_EFFICIENCY)
log_x = np.log10(THROUGHPUT[1:])
log_y = np.log10(ENERGY_EFFICIENCY[1:])
coefs3 = np.polyfit(log_x, log_y, 1)
y_pred = coefs3[0] * log_x + coefs3[1]
ss_res = float(np.sum((log_y - y_pred) ** 2))
ss_tot = float(np.sum((log_y - log_y.mean()) ** 2))
r2_e  = 1 - ss_res / ss_tot
xs = np.logspace(np.log10(THROUGHPUT[1:].min()), np.log10(THROUGHPUT[1:].max()), 100)
ax.plot(xs, 10**(coefs3[0]*np.log10(xs) + coefs3[1]), 'b--', linewidth=0.4, alpha=0.8)
ax.text(0.04, 0.96,
        f'slope = {coefs3[0]:.2f}\n$R^2$ = {r2_e:.3f}',
        transform=ax.transAxes, fontsize=6, va='top', ha='left',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                  edgecolor='none', alpha=0.75))
ax.grid(True, which='both', linestyle='--', linewidth=0.2)

# Legends -------------------------------------------------------------------
algo_names  = ['CLP', 'NCM', 'Replay', 'SLDA']
algo_colors = [p[9], p[2], p[3], p[1]]
algo_handles = [Line2D([0],[0], marker='o', color='w',
                       markerfacecolor=c, markersize=5, label=n)
                for c, n in zip(algo_colors, algo_names)]
arch_handles = [
    Line2D([0],[0], marker='*', color='w', markerfacecolor='black',
           markersize=10, label='Loihi 2'),
    Line2D([0],[0], marker='D', color='w', markerfacecolor='black',
           markersize=5,  label='GPU (Nvidia\nJetson Orin Nano)'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='black',
           markersize=5,  label='CPU (6-core Arm\nCortex-A78AE)'),
]
main_legend = axes[0].legend(handles=algo_handles, title='OCL Algorithms',
                             fontsize=7, loc='lower left',
                             bbox_to_anchor=(-0.02, 0))
axes[0].add_artist(main_legend)
axes[1].legend(handles=arch_handles, title='Reference Architectures',
               fontsize=7, loc='lower left', bbox_to_anchor=(-0.02, 0))

plt.tight_layout()
for ext in ('pdf', 'png', 'svg'):
    plt.savefig(IMAGES_DIR / f'pareto_plots.{ext}', format=ext, bbox_inches='tight')
plt.show()
print('Saved images/pareto_plots.{pdf,png,svg}')
print(f'Panel E log-log fit: slope = {coefs3[0]:.3f}, '
      f'intercept = {coefs3[1]:.3f}, R^2 = {r2_e:.4f}')
